Input: 
1. Excel file containing news articles generated by scraping news pages. How to generate this web-scraped CSV: Cutoff date - March 1 2024
2. What period to analyze the company data from-to.

Output:
Sentiment analysis for news articles into Positive, Neutral, and Negative.
Identify earnings date for each company.
Categorize companies at a quarter level with positive sentiments, negative sentiments prior to earnings date.

TBD: Categorize reports by quarter.
TBD: Identify contrarian viewpoints. at quarter level or year level?
TBD: Have mechanism to qualify whether contrarian views were actually correct in retrospect.



In [ ]:
import pandas as pd

# Using manual dataframe instead of Langchain doc loader as need to do processing below.
scraped_articles_df = pd.read_excel("./data/scraped_articles.xlsx", dtype=str)

# Since Excel has a limit of 32k on a cell, concatenate actual article body from multiple cells.
scraped_articles_df["article_body"] = scraped_articles_df.article_body.map(str) + \
    str(scraped_articles_df.article_body_2) + str(scraped_articles_df.article_body_3)

# Drop redundant columns now that we have all article bodies in "article_body" already.
scraped_articles_df = scraped_articles_df.drop(columns=["article_body_2", "article_body_3"])


print(f"Length of scraped articles: {len(scraped_articles_df)}" )
print(f"Companies: {scraped_articles_df["company_name"].unique()}" )
print(f"Number of authors: {len(scraped_articles_df["author"].unique())}" )
print(f"Websites: {scraped_articles_df["source_name"].unique()}" )
print(f"Websites count: {len(scraped_articles_df["source_name"].unique())}" )
print(f"Already analzyed articles count: {len(scraped_articles_df) - len(scraped_articles_df["analyzed_sentiment"].isna())}" )

print("--Dataframe info below\n--\n")
print(scraped_articles_df.info())
print("\n--Dataframe info end --\n")

Length of scraped articles: 56
Companies: ['Apple']
Number of authors: 33
Websites: ['Wired' 'The Verge' 'Gizmodo.com']
Websites count: 3
Already analzyed articles count: 0
--Dataframe info below
--

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56 entries, 0 to 55
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   source_name         56 non-null     object
 1   author              56 non-null     object
 2   company_name        56 non-null     object
 3   title               56 non-null     object
 4   description         56 non-null     object
 5   published           56 non-null     object
 6   url                 56 non-null     object
 7   article_body        56 non-null     object
 8   analyzed_sentiment  0 non-null      object
dtypes: object(9)
memory usage: 4.1+ KB
None

--Dataframe info end --



In [ ]:
## Debug cell
# print(scraped_articles_df.iloc[0,:])

source_name                                                       Wired
author                                                      Adrienne So
company_name                                                      Apple
title                 The Apple Watch Turns 10. Here's How Far It's ...
description           When the Apple Watch launched, it was unclear ...
published                                          2025-04-24T14:02:22Z
url                   https://www.wired.com/story/apple-watch-turns-10/
article_body          It’s been 10years since theApple Watchdebuted....
analyzed_sentiment                                                  NaN
Name: 0, dtype: object


In [29]:
import langchain
import os
from getpass import getpass
from langchain_openai import ChatOpenAI

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY") or getpass(
    "Enter OpenAI API Key: "
)

openai_model = "gpt-4o-mini"

llm = ChatOpenAI(temperature=0.0, model=openai_model)

In [ ]:
from langchain.prompts import (
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    ChatPromptTemplate
)

# Reduce set to test, comment later to process across all
articles_dataframe = scraped_articles_df[1:5].copy()

def process_row_sentiment(article_row):
    system_prompt = """You are a market research analyst for Berkshire Hathway who has to help the user analyze the sentiment for the user provided news 
    article about a company.
    The user will use it to analyse current market sentiment for the company, and will consider buying stocks based on your analysis.
    Respond ONLY with one among the following sentiments - Positive, Negative, Neutral or "N/A"
    If unsure about the sentiment of the article, or you fail to see how the news article may affect the market sentiment of a company, respond with "N/A".
    Do not respond with anything other than the aforementioned sentiments.
    Only consider the news article to form your judgement of the sentiment of the article.
    """

    user_prompt = """Help me analyze the sentiment for the following news article - 
    Title: {title} ,
    Description: {description},
    URL: {url}, 
    Article body: {article_body}, 
    """

    prompt_template = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("user", user_prompt),
    ])

    # print(prompt_template.input_variables)

    pipeline = prompt_template | llm
    response = pipeline.invoke(
        {
            "title": article_row["title"], 
            "description": article_row["description"],
            "url": article_row["url"], 
            "article_body": article_row["article_body"]
        })
    print(article_row["title"])
    print(response)


for _, article_row in articles_dataframe.iterrows():
    # print(article_row)
    sentiment = process_row_sentiment(article_row)

22 Best MacBook Accessories (2025), Tested and Reviewed
content='Neutral' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 1, 'prompt_tokens': 2386, 'total_tokens': 2387, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_54eb4bd693', 'id': 'chatcmpl-BbJpVaInarHEeV6LxVe335QImeAoU', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='run--3f40a170-eb01-4817-b021-b17aeefbdb37-0' usage_metadata={'input_tokens': 2386, 'output_tokens': 1, 'total_tokens': 2387, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
How to Use Apple Maps on the Web
content='Positive' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 1,